# Démo — Affiner un petit modèle génératif sur des épigrammes

Ce notebook permet de :

- observer le format des données d'entraînement ;
- comprendre ce que le modèle reçoit réellement ;
- lancer ou simuler un petit entraînement ;
- générer un texte avant et après affinage, si un checkpoint est disponible. 

> Pour une démonstration, le paramètre `RUN_TRAINING` est mis à `False` par défaut.  
> On peut le passer à `True` si l'environnement est prêt et si le temps le permet.

## 1. Installer et importer les bibliothèques

Nous utilisons Hugging Face `transformers` pour le modèle et `datasets` pour préparer le corpus.

Le modèle de démonstration est `distilgpt2`, plus léger que `gpt2`.  
On peut revenir au modèle original en remplaçant `MODEL_NAME = "distilgpt2"` par `MODEL_NAME = "gpt2"`.

In [1]:
# Dans un environnement neuf, décommenter au besoin :
# !pip install transformers datasets torch pandas

from pathlib import Path
from datetime import datetime

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

PyTorch : 2.5.1+cu124
CUDA disponible : False


## 2. Configuration de la démo

Quelques paramètres importants :

- `MODEL_NAME` : le modèle de départ ;
- `DATA_FILE` : le fichier d'épigrammes formatées, si disponible ;
- `RUN_TRAINING` : lance réellement l'entraînement si la valeur est `True` ;
- `MAX_STEPS` : limite le nombre d'étapes pour éviter une démo interminable.

In [27]:
MODEL_NAME = "distilgpt2"      
DATA_FILE = Path("./corpus/formatted_epigrams_eng_genre.txt")
OUTPUT_DIR = Path("./checkpoints/epigram_gpt2_demo")
FINAL_MODEL_DIR = Path("./gpt2-epigram-demo-final")

RUN_TRAINING = True            # Passer à True pour entraîner réellement
MAX_STEPS = 50                 # Petit nombre pour une démo
BLOCK_SIZE = 128                # Taille des blocs de tokens
SEED = 42

PROMPT_TEMPLATE = "<author>Anonymous</author>\n<genre>funerary</genre>\n<epigram>\n"

## 3. Le format des données

Le notebook original utilise un format balisé :

```text
<author>Anonymous</author>
<genre>erotic</genre>
<epigram>
...
<end>
```

Ce format est très utile pédagogiquement : il montre au modèle un patron régulier.

Le modèle n'apprend pas seulement à produire du texte.  
Il apprend à continuer un texte dans un format précis, avec des indices comme l'auteur et le genre.

In [3]:
if DATA_FILE.exists():
    raw_text = DATA_FILE.read_text(encoding="utf-8")
    texts = [t.strip() + "\n<end>" for t in raw_text.split("<end>") if t.strip()]
    source = f"fichier trouvé : {DATA_FILE}"

print(f"Source des données : {source}")
print(f"Nombre d'exemples : {len(texts)}")
print("\nPremier exemple :\n")
print(texts[0])

Source des données : fichier trouvé : corpus/formatted_epigrams_eng_genre.txt
Nombre d'exemples : 3135

Premier exemple :

<author>anonymous</author>
<genre>ecphrastique</genre>
<epigram>
Inscribed on the Tabernacle of Saint Sophia

The images that the hereties took down here our pious sovereigns replaced.
</epigram>
<end>


## 4. Mettre les exemples dans un tableau

Avant l'entraînement, on inspecte toujours les données.

Un modèle affiné apprend à partir de ce qu'on lui donne.  
Si le corpus est flou, incohérent ou mal formaté, le modèle apprendra ce flou avec beaucoup d'assurance.  
Le chaos, mais vectorisé.

In [4]:
df = pd.DataFrame({"text": texts})
df["length_chars"] = df["text"].str.len()
df.head()

,text,length_chars
0,<author>anonymous</author>\n<genre>ecphrastiqu...,201
1,<author>anonymous</author>\n<genre>ecphrastiqu...,359
2,<author>anonymous</author>\n<genre>ecphrastiqu...,261
3,<author>anonymous</author>\n<genre>ecphrastiqu...,310
4,<author>anonymous</author>\n<genre>ecphrastiqu...,426


## 5. Charger le tokenizer et le modèle

Le tokenizer transforme le texte en identifiants numériques.  
Le modèle reçoit ces identifiants, pas les mots eux-mêmes.

Ici, nous utilisons un modèle de langage causal : il apprend à prédire le token suivant.

In [19]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 et DistilGPT-2 n'ont pas de pad token par défaut.
# On réutilise donc le token de fin de texte comme token de padding.
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.loss_type = "ForCausalLM"

print("Modèle chargé :", MODEL_NAME)
print("Taille du vocabulaire :", tokenizer.vocab_size)

Modèle chargé : distilgpt2
Taille du vocabulaire : 50257


## 6. Observer la tokenisation

Regardons ce que devient un exemple une fois passé dans le tokenizer.

Les balises `<author>`, `<genre>`, `<epigram>` et `<end>` ne sont pas magiques.  
Pour le modèle, ce sont seulement des morceaux de texte qui reviennent souvent.

In [6]:
example_text = texts[0]
encoded_example = tokenizer(example_text)
tokens = tokenizer.convert_ids_to_tokens(encoded_example["input_ids"])

pd.DataFrame({
    "position": range(len(tokens)),
    "token": tokens,
    "id": encoded_example["input_ids"],
}).head(80)

,position,token,id
0,0,<,27
1,1,author,9800
2,2,>,29
3,3,an,272
4,4,onymous,6704
...,...,...,...
58,58,>,29
59,59,Ċ,198
60,60,<,27
61,61,end,437


## 7. Créer un Dataset Hugging Face

On convertit maintenant notre liste de textes en objet `Dataset`.

C'est le format attendu par `Trainer`.

In [7]:
dataset = Dataset.from_dict({"text": texts})
dataset

Dataset({
    features: ['text'],
    num_rows: 3135
})

## 8. Tokeniser tout le corpus

On applique le tokenizer à tous les exemples.

À cette étape, le texte devient une suite d'identifiants numériques.  
C'est le matériau d'entraînement du modèle.

In [8]:
block_size = 1024

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=block_size,
        return_special_tokens_mask=True
    )

tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized

Map:   0%|          | 0/3135 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 3135
})

## 9. Grouper les tokens en blocs

Le notebook original concaténait les exemples puis les divisait en blocs de taille fixe.

C'est une stratégie fréquente pour l'entraînement de modèles de langage :  
le modèle apprend à prédire la suite dans des séquences de longueur régulière.

In [9]:
def group_texts(examples):
    # On concatène tous les exemples du batch.
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])

    # On coupe ce qui dépasse pour obtenir des blocs complets.
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE

    result = {
        k: [t[i:i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    return result

lm_dataset = tokenized.map(group_texts, batched=True, batch_size=1000)

print(lm_dataset)
print("Nombre de blocs :", len(lm_dataset))

Map:   0%|          | 0/3135 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 2599
})
Nombre de blocs : 2599


## 10. Le collator : préparer les lots d'entraînement

Le `DataCollatorForLanguageModeling` prépare les lots que le modèle reçoit.

Ici, `mlm=False` parce que GPT-2 est un modèle de langage causal : il prédit la suite du texte.  
Ce n'est pas le même objectif que BERT, qui utilise le masquage de tokens pendant son pré-entraînement.

In [10]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

batch = data_collator([lm_dataset[i] for i in range(min(2, len(lm_dataset)))])

print("input_ids :", batch["input_ids"].shape)
print("labels    :", batch["labels"].shape)

input_ids : torch.Size([2, 128])
labels    : torch.Size([2, 128])


## 11. Générer avant l'affinage

Avant de modifier le modèle, on peut lui demander de compléter le prompt.

Cela donne un point de comparaison.  
Le modèle de base connaît l'anglais en général, mais il ne connaît pas encore notre petit format d'épigrammes.

In [18]:
def generate_text(model, tokenizer, prompt, max_new_tokens=80, temperature=0.9, top_p=0.92, top_k=40):
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)

    output = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

print(generate_text(model, tokenizer, PROMPT_TEMPLATE))

<author>Anonymous</author>
<genre>funerary</genre>
<epigram>
This is not the day when all the time is spent on this land and is being spent on the land of the dead. But this is not. Why do thou have no land, and why have no land to feed on, in spite of all the good oases of the old land. Let me ask, ‑is that he will go to war with the poor, and when did he


## 12. Préparer l'entraînement

In [22]:
batch_size = 4 if torch.cuda.is_available() else 2

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    overwrite_output_dir=True,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=2,
    save_steps=MAX_STEPS,
    save_total_limit=1,
    logging_steps=5,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=data_collator,
)

print("Configuration prête.")
print("RUN_TRAINING =", RUN_TRAINING)

Configuration prête.
RUN_TRAINING = True


## 13. Lancer ou simuler l'entraînement

In [23]:
if RUN_TRAINING:
    trainer.train()
    trainer.save_model(str(FINAL_MODEL_DIR))
    tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
    print(f"Modèle sauvegardé dans : {FINAL_MODEL_DIR}")
else:
    print("Entraînement non lancé.")
    print("Pour entraîner réellement, mettre RUN_TRAINING = True puis relancer les cellules d'entraînement.")

Step,Training Loss
5,4.369200
10,4.040000
15,3.621100
20,3.696200
25,3.366000
30,3.276900
35,3.787100
40,3.361200
45,3.509200
50,3.230800


Modèle sauvegardé dans : gpt2-epigram-demo-final


## 14. Générer après affinage, si un modèle existe

Si un modèle affiné a été sauvegardé, on le recharge et on compare sa génération à celle du modèle de base.

Sinon, la cellule explique simplement qu'aucun modèle affiné n'est disponible.

In [25]:
if FINAL_MODEL_DIR.exists():
    fine_tuned_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_MODEL_DIR))
    fine_tuned_tokenizer.pad_token = fine_tuned_tokenizer.eos_token
    fine_tuned_model = AutoModelForCausalLM.from_pretrained(str(FINAL_MODEL_DIR))

    print("=== Modèle affiné ===")
    print(generate_text(fine_tuned_model, fine_tuned_tokenizer, PROMPT_TEMPLATE))
else:
    print(f"Aucun modèle affiné trouvé dans {FINAL_MODEL_DIR}.")
    print("Lancez l'entraînement pour produire un modèle comparatif.")

=== Modèle affiné ===
<author>Anonymous</author>
<genre>funerary</genre>
<epigram>
My father was the most important figure of the world. I was the chief of the Church. He sent his own sons from here to me, and even from here, to the city of Athens. My father was the priest, and his father was the priest. My father, by nature, was the holy god, the most holy god. My father was the king of the world. I was


## 15. Comparer plusieurs genres

Une fois le modèle affiné disponible, on peut tester si les balises de genre influencent la génération.

C'est exactement le type de comportement qu'un fine-tuning peut apprendre :  
répondre de manière plus stable à une structure d'entrée récurrente.

In [26]:
genres = ["funéraire", "érotique", "votive"]

def make_prompt(genre, author="Anonymous"):
    return f"<author>{author}</author>\n<genre>{genre}</genre>\n<epigram>\n"

model_to_use = fine_tuned_model if "fine_tuned_model" in globals() else model
tokenizer_to_use = fine_tuned_tokenizer if "fine_tuned_tokenizer" in globals() else tokenizer

for genre in genres:
    prompt = make_prompt(genre)
    print("\n" + "=" * 80)
    print("PROMPT :", repr(prompt))
    print(generate_text(model_to_use, tokenizer_to_use, prompt, max_new_tokens=70))


PROMPT : '<author>Anonymous</author>\n<genre>funéraire</genre>\n<epigram>\n'
<author>Anonymous</author>
<genre>funéraire</genre>
<epigram>
I used to play in the city, and I am not afraid to call it home.
</epigram>
<end><author>Pascal de Bress</author>
<genre>funéraire</genre>
<epigram>
I know that you have to take care of yourself. If you

PROMPT : '<author>Anonymous</author>\n<genre>érotique</genre>\n<epigram>\n'
<author>Anonymous</author>
<genre>érotique</genre>
<epigram>
You were one of the only persons of the city of Orpheus to enter the Aegean to enter the Aegean. But the way of taking a journey, and entering the Aegean in a manner that is pleasing to you, was this in order to be a part of a city of a kind and worthy of your patronage. What

PROMPT : '<author>Anonymous</author>\n<genre>votive</genre>\n<epigram>\n'
<author>Anonymous</author>
<genre>votive</genre>
<epigram>
In the midst of the flood of rain, a large wave of fire rises, and is carried out by this stream of fire, a 